# Dazo — ChatGPT ↔ Colab bridge v2

Run the cell below once in a GPU runtime. It mounts Google Drive, safely updates `8dazo/dazo` without deleting untracked checkpoints/data, installs Dazo, and starts the bounded task bridge.

Bridge v2 polls commands with `git fetch + git show` instead of GitHub raw-file CDN polling and writes a Drive heartbeat every ~10 seconds. Leave the cell running. Results/logs are written to `MyDrive/DazoBridge`. No SSH/tunnel or GitHub write token is used.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess

drive.mount('/content/drive')
os.chdir('/content')
repo = Path('/content/dazo')

if (repo / '.git').exists():
    subprocess.run(['git', '-C', str(repo), 'fetch', '--depth=1', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/main'], check=True)
elif repo.exists():
    backup = Path('/content/dazo-pre-bridge')
    if backup.exists():
        shutil.rmtree(backup)
    repo.rename(backup)
    subprocess.run(['git', 'clone', '-q', 'https://github.com/8dazo/dazo.git', str(repo)], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/8dazo/dazo.git', str(repo)], check=True)

os.chdir(repo)
print('repo:', Path.cwd())
print('commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[train]'], check=True)
print('\nStarting Dazo bridge v2. Leave this cell running.\n')
subprocess.run(['python', '.colab/bridge_v2.py'], check=True)


## What happens next

Once the cell prints `Dazo bridge v2 starting` and `Dazo bridge online`, return to ChatGPT and say **bridge v2 online**. ChatGPT can then queue bounded Dazo tasks through `.colab/command.json` and read results/heartbeat from Drive.
